In [ ]:
# # Unsloth 설치 및 최신 버전 업그레이드
# !pip install unsloth
# !pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install trl bitsandbytes triton xformers peft

# # GPU가 softcapping을 지원하는 경우, Flash Attention 2 설치
# import torch
# if torch.cuda.get_device_capability()[0] >= 8:
#     !pip install --no-deps packaging ninja einops "flash-attn>=2.6.3"

# 1. 모델 로드

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None, # None 인경우 컴퓨팅 유닛에 알맞는 dtype으로 설정됨
    load_in_4bit = True, # 4bit 양자화
    token = "hf ..." # huggingface Token 입력
)

# 2. QLoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['p_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.1,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = {"quantization": "4bit"},
)

# 3. 데이터 셋 EOS 추가

In [ ]:
from datasets import load_dataset

EOS_TOKEN = tokenizer.eos_token
dataset = load_dataset("csv", data_files='datasets/formatted_finance.csv', split="train")

def add_eos_token(examples):
    examples['formatted_text'] = examples["formatted_text"] + EOS_TOKEN
    return examples

dataset = dataset.map(add_eos_token)


# 4. 모델 학습

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "formatted_text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001
    )
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "formatted_text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # True로 설정하면 짧은 텍스트 데이터에 대해서는 더 빠른 학습 속도로를 보여줍니다.
    args = TrainingArguments( # TrainingArguments는 자신의 학습 환경과 기호에 따라 적절하게 설정하면 됩니다.
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        # max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "/krx/save_steps",
        save_steps=100,  # 매 100 스텝마다 체크포인트 저장
        save_total_limit=1,  # 최신 1개의 체크포인트만 보관
    ),
)